<a href="https://colab.research.google.com/github/K415mm/mcp_course_workshops/blob/main/Workshop_04_Malware_Analysis/04_Malware_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦠 Workshop 4: Malware Analysis with MCP

Welcome to the fourth workshop of the RAISEGUARD Advanced Agentic AI Course!

**Goal:** Build a safe malware analysis workflow using MCP: detect file type, extract static indicators (hashes, strings, PE imports), normalize output to prevent LLM bias, and produce a neutral analyst brief.

### Setup & Libraries
Run the cell below to install our dependencies.

In [ ]:
!pip install groq langchain-groq mcp requests -q
print("✅ Libraries installed successfully!")

### API Keys
For this workshop, you will need:
1. **Groq API Key** (For our LLM Brain)
2. **VirusTotal API Key** (For File Hash checking)

Add them to the Google Colab **Secrets (🔑)** tab on the left menu, then run this cell.

In [ ]:
import os
from google.colab import userdata

keys = ["GROQ_API_KEY", "VT_API_KEY"]
for k in keys:
    try:
        os.environ[k] = userdata.get(k)
        print(f"✅ {k} successfully loaded!")
    except userdata.SecretNotFoundError:
        print(f"⚠️ WARNING: {k} missing from Colab Secrets.")
        os.environ[k] = "" 

--- 
## 💾 Lab Data Setup: Synthetic File Creation
**SAFETY FIRST: We are using a synthetic safe file, not real malware.**
Run this python script below to generate our `sample_update.bin` file with embedded strings to simulate a Windows Executable.

In [ ]:
output_path = "sample_update.bin"

# Synthetic file with PE magic bytes and typical interesting strings
content = b"MZ" + b"\x90" * 58 + b"PE\x00\x00"  # PE header magic
content += b"\x00" * 100
content += b"VirtualAlloc\x00CreateRemoteThread\x00WriteProcessMemory\x00"
content += b"LoadLibraryA\x00GetProcAddress\x00"
content += b"http://update.example.net/payload\x00"
content += b"cmd.exe /c whoami\x00"
content += b"SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\x00"
content += b"\x00" * 100

with open(output_path, "wb") as f:
    f.write(content)

print(f"✅ Sample file created: {output_path} ({len(content)} bytes)")

--- 
## 🟢 Beginner Profile: Building the Static Analysis Scripts
Before we build an autonomous agent, we need to build string extraction logic to look for C2 URLs and malicious Windows APIs inside the binary.

In [ ]:
import hashlib
import re

FILE_PATH = "sample_update.bin"

def compute_file_hashes(file_path: str) -> dict:
    """Compute MD5, SHA1, and SHA256 hashes for a file."""
    try:
        with open(file_path, "rb") as f:
            data = f.read()
        return {
            "md5": hashlib.md5(data).hexdigest(),
            "sha256": hashlib.sha256(data).hexdigest(),
            "size_bytes": len(data),
            "status": "ok"
        }
    except Exception as e:
        return {"status": "error", "reason": str(e)}

def detect_file_type(file_path: str) -> dict:
    """Detect the type of a file by reading its magic bytes (file signature)."""
    try:
        with open(file_path, "rb") as f:
            data = f.read(512)
        if data[:2] == b"MZ":
            file_type = "Windows PE Executable"
        elif data[:4] == b"%PDF":
            file_type = "PDF Document"
        elif data[:4] in (b"PK\x03\x04", b"PK\x05\x06"):
            file_type = "ZIP/Office Archive"
        else:
            file_type = "Unknown"
        return {"detected_type": file_type, "status": "ok"}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

# Let's test it!
print("File Identity:\n", json.dumps(compute_file_hashes(FILE_PATH), indent=2))
print("Detected Format:\n", detect_file_type(FILE_PATH))

--- 
## 🟡 Intermediate Profile: Extract Strings via FastMCP Toolkit
Now we build our advanced String Extraction tools and bind them all to the MCP Server so the LLM can use them.

In [ ]:
from mcp.server.fastmcp import FastMCP
import requests

mcp = FastMCP("Malware Analysis Server")
VT_KEY = os.environ.get("VT_API_KEY", "")

@mcp.tool()
def mcp_compute_hashes() -> dict:
    """Compute hashes on the local sample binary."""
    return compute_file_hashes(FILE_PATH)

@mcp.tool()
def extract_strings(min_length: int = 6) -> dict:
    """Extract printable ASCII strings from the local file to find URLs, registry keys, and commands."""
    try:
        with open(FILE_PATH, "rb") as f:
            data = f.read()
        pattern = rb"[ -~]{" + str(min_length).encode() + rb",}"
        raw_strings = re.findall(pattern, data)
        decoded = [s.decode("ascii", errors="ignore") for s in raw_strings]
        
        urls = [s for s in decoded if s.startswith("http")]
        cmd_strings = [s for s in decoded if any(k in s.lower() for k in ["cmd", "powershell", "whoami"])]
        api_strings = [s for s in decoded if any(a in s for a in ["VirtualAlloc", "WriteProcess"])]
        
        return {
            "urls_found": urls,
            "command_strings": cmd_strings,
            "notable_api_references": api_strings,
            "status": "ok"
        }
    except Exception as e:
        return {"status": "error", "reason": str(e)}

@mcp.tool()
def enrich_hash(file_hash: str) -> dict:
    """Look up a file hash on VirusTotal."""
    try:
        attrs = requests.get(
            f"https://www.virustotal.com/api/v3/files/{file_hash}",
            headers={"x-apikey": VT_KEY}, timeout=10
        ).json().get("data", {}).get("attributes", {})
        stats = attrs.get("last_analysis_stats", {})
        return {"hash": file_hash, "malicious_detections": stats.get("malicious", 0)}
    except Exception as e:
        return {"status": "error", "reason": "Not Found in DB or Error"}

print("✅ FastMCP Server initialized with tools:", [t.name for t in mcp._tool_manager.get_tools()])

--- 
## 🔴 Advanced Profile: The Autonomous Malware Analyst
We now provide the agent with our entire toolkit and command it to autonomously reverse engineer the file and write a brief!

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

# Langchain compatible tools
@tool
def ai_compute_hashes(query: str) -> str:
    """Compute hashes of the suspicious file. Query input can be 'all'."""
    return str(mcp_compute_hashes())

@tool
def ai_extract_strings(query: str) -> str:
    """Extract embedded URLs, commands, and APIs from the file. Query input can be 'all'."""
    return str(extract_strings())

@tool
def ai_enrich_hash(file_hash: str) -> str:
    """Pass a SHA256 Hash to VirusTotal to get its threat score."""
    return str(enrich_hash(file_hash))

tools = [ai_compute_hashes, ai_extract_strings, ai_enrich_hash]

agent = initialize_agent(
    tools, llm, agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

print("🚀 Agent Pipeline Ready!")

In [ ]:
prompt = f"""
We received a suspicious file named sample_update.bin from an alert.
Perform a safe static analysis using your tools:
1. Compute its hashes.
2. Look up the SHA256 hash against threat intel using your tool.
3. Extract strings from the file to find URLs, Commands, and APIs.
4. Produce a MALWARE ANALYSIS BRIEF with VERDICT, FILE IDENTITY, STATIC FINDINGS, and RECOMMENDED ACTIONS.
"""

# Kick off the autonomous loop!
agent.run(prompt)